## Dịch gold NER VI → EN (Qwen qua Hugging Face)

Điều khiển pipeline chuẩn trong `source/translation/`: clone/cập nhật repo,
mount Drive, gọi `source.translation.pipeline.run_pipeline` (model tải từ
Hugging Face bằng `transformers`), rồi kiểm chứng file đã ghi.

Notebook **không chứa logic pipeline**: gắn tag, prompt, parse, validate span,
resume và ghi output đều nằm trong `source/translation/`.

File output giữ **đúng schema của input**: `entities[].vi_term`,
`vi_start_token_idx`, `vi_end_token_idx` và `vi_sentence_str` — nhưng mang giá
trị tiếng Anh, với token index tính lại trên câu dịch sao cho
`" ".join(vi_sentence_str.split()[start:end + 1]) == vi_term`.

Output nằm cạnh `OUTPUT_PATH`: `translated.jsonl` (pass), `review.jsonl`
(fail), `translated.jsonl.done.jsonl` (manifest resume).

In [ ]:
%pip install "transformers>=4.44" "accelerate>=0.34" hf_transfer

## Import Libraries

Chỉ thư viện chuẩn ở đây; package `source.translation` được import sau khi
repo đã có trên runtime.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
from pathlib import Path

# Phải set TRƯỚC khi huggingface_hub được import (nó được import trong
# HFBackend lúc Stage 3). Mặc định hub dùng Xet: tải xong còn "Reconstructing"
# từng chunk, rất chậm với file safetensors 60GB. Tắt Xet + bật hf_transfer
# (Rust, tải song song nhiều kết nối) nhanh hơn nhiều.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

print("Python:", sys.version.split()[0])

## Constants and Configuration

Sửa `DRIVE_DATA_DIR` một lần; mọi đường dẫn Drive suy ra từ đó.

`MODEL_ID` là repo id trên Hugging Face; `AutoModelForCausalLM` tự tải weights
về Colab ở lần chạy đầu (cache ở `~/.cache/huggingface`). Không cần copy model
lên Drive. Mỗi runtime mới sẽ tải lại ~60GB, nên cell Import đã tắt Xet và bật
`hf_transfer` cho nhanh — đừng xoá mấy dòng env đó.

`RESUME = True` bỏ qua record đã có trong output + manifest; đặt `False` để
truncate cả ba file output và chạy lại từ đầu. `LIMIT` chỉ để smoke test.

In [ ]:
# ---------------------------------------------------------------------------
# Git repository
# ---------------------------------------------------------------------------
GIT_REPO_URL = "https://github.com/minhhhduc/bio-linking-.git"
GIT_BRANCH = "main"
REPO_CLONE_DIR = Path("/content/bio_linking")

# ---------------------------------------------------------------------------
# Google Drive (REQUIRED: set DRIVE_DATA_DIR below)
# ---------------------------------------------------------------------------
MOUNT_DRIVE = True
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_DATA_DIR = DRIVE_MOUNT_POINT / "MyDrive" / "work" / "nlp" / "UMLS" / "entity_linking" / "data"

# Inputs (must already exist)
INPUT_PATH = DRIVE_DATA_DIR / "test_fixed_verified_gold_revised.jsonl"

# Translation LLM: Hugging Face repo id, transformers tự tải weights về Colab
MODEL_ID = "Qwen/Qwen3.8-27B"            # đổi sang repo id bạn muốn chạy
DTYPE = "bfloat16"                     # A100 native; float16 | float32 optional
MAX_SEQ_LEN = None                     # None = model max_position_embeddings
MAX_NEW_TOKENS = 2048                  # output cap; generation stops at EOS

# Outputs (OUTPUT_DIR must already exist)
OUTPUT_DIR = DRIVE_DATA_DIR
OUTPUT_PATH = OUTPUT_DIR / "translated.jsonl"
REVIEW_PATH = OUTPUT_DIR / "review.jsonl"

# ---------------------------------------------------------------------------
# Pipeline settings (A100 80GB)
# ---------------------------------------------------------------------------
BATCH_SIZE = 32
MAX_RETRIES = 2
LIMIT = None            # None = all records; an int truncates for a smoke run
RESUME = True           # False truncates output, review and manifest

print(f"Input  : {INPUT_PATH}")
print(f"Model  : {MODEL_ID} ({DTYPE}, tải từ Hugging Face)")
print(f"Limits : max_seq_len={MAX_SEQ_LEN or 'model max'}, max_new_tokens={MAX_NEW_TOKENS}, batch={BATCH_SIZE}")
print(f"Output : {OUTPUT_PATH}")

## Stage 1 — Clone or Update the Repository

Package dịch nằm trong repo (`source/translation/`), không nằm trong notebook.
Chạy lại cell này sẽ cập nhật bản clone thay vì tải lại.

In [ ]:
def _git(*args: str) -> None:
    print("git", " ".join(args))
    subprocess.run(["git", *args], check=True)


if (REPO_CLONE_DIR / ".git").exists():
    _git("-C", str(REPO_CLONE_DIR), "fetch", "--depth", "1", "origin", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "checkout", GIT_BRANCH)
    _git("-C", str(REPO_CLONE_DIR), "pull", "--ff-only", "origin", GIT_BRANCH)
else:
    REPO_CLONE_DIR.parent.mkdir(parents=True, exist_ok=True)
    _git("clone", "--depth", "1", "--branch", GIT_BRANCH, GIT_REPO_URL, str(REPO_CLONE_DIR))

if not (REPO_CLONE_DIR / "source" / "translation" / "pipeline.py").is_file():
    raise FileNotFoundError(
        f"source/translation/pipeline.py is missing from {REPO_CLONE_DIR}; "
        "check GIT_BRANCH and the repository layout."
    )
if str(REPO_CLONE_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_CLONE_DIR))

from source.translation.pipeline import PipelineConfig, run_pipeline

print("Repository ready:", REPO_CLONE_DIR)

## Stage 2 — Mount Drive and Validate the Paths

Input và output dir phải có sẵn trên Drive; notebook không tạo folder. Model
được tải từ Hugging Face ở Stage 3, không cần path local.

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))


def require_path(path, name: str) -> Path:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    return path


input_path = require_path(INPUT_PATH, "INPUT_PATH")
output_dir = require_path(OUTPUT_DIR, "OUTPUT_DIR")
if not output_dir.is_dir():
    raise NotADirectoryError(f"OUTPUT_DIR is not a directory: {output_dir}")

for label, path in (("OUTPUT_PATH", OUTPUT_PATH), ("REVIEW_PATH", REVIEW_PATH)):
    if Path(path).parent.resolve() != output_dir.resolve():
        raise ValueError(f"{label} must live in OUTPUT_DIR, got {Path(path).parent}")

print(f"Input records : {sum(1 for line in input_path.open(encoding='utf-8-sig') if line.strip()):,}")
print(f"Model         : {MODEL_ID} (tải từ Hugging Face khi chạy Stage 3)")

## Stage 3 — Run the Pipeline

`run_pipeline` đọc record, gắn tag, dịch theo batch, retry khi validation fail,
và ghi `translated.jsonl` / `review.jsonl` / manifest ngay khi từng batch xong.
Ngắt runtime rồi chạy lại với `RESUME = True` sẽ tiếp tục, không dịch lại.

In [ ]:
config = PipelineConfig(
    model=MODEL_ID,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    max_seq_len=MAX_SEQ_LEN,
    dtype=DTYPE,
    input_path=str(input_path),
    output_path=str(OUTPUT_PATH),
    review_path=str(REVIEW_PATH),
    max_retries=MAX_RETRIES,
    resume=RESUME,
    limit=LIMIT,
)

started = time.time()
stats = run_pipeline(config)
print(json.dumps(stats.as_dict(), ensure_ascii=False, indent=2))
print(f"Wall clock: {time.time() - started:.1f}s")

## Stage 4 — Verify the Written Files

Output phải **cùng schema với input**: mỗi dòng có `vi_sentence_str` và
`entities[]` với `cui`, `icd10`, `label`, `vi_term`, `vi_start_token_idx`,
`vi_end_token_idx`. Kiểm tra giá trị đã là tiếng Anh và span trỏ đúng chỗ:
`" ".join(sentence.split()[start:end + 1]) == term` (token whitespace, đúng
quy ước của `source/common/data.py`).

Record fail nằm trong `review.jsonl`; chạy lại với `RESUME = True` sẽ thử lại
chúng, vì review không được dùng để skip.

In [ ]:
ENTITY_KEYS = {
    "cui", "icd10", "label", "vi_term", "vi_start_token_idx", "vi_end_token_idx"
}


def read_jsonl(path) -> list[dict]:
    path = Path(path)
    if not path.is_file():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


translated = read_jsonl(OUTPUT_PATH)
review = read_jsonl(REVIEW_PATH)

for line_no, row in enumerate(translated, 1):
    assert set(row) == {"entities", "vi_sentence_str"}, (line_no, set(row))
    tokens = row["vi_sentence_str"].split()
    for entity in row["entities"]:
        assert set(entity) == ENTITY_KEYS, (line_no, set(entity))
        start = entity["vi_start_token_idx"]
        end = entity["vi_end_token_idx"]
        assert 0 <= start <= end < len(tokens), (line_no, start, end, len(tokens))
        assert " ".join(tokens[start:end + 1]) == entity["vi_term"], (
            line_no,
            entity["vi_term"],
            " ".join(tokens[start:end + 1]),
        )

print(f"translated: {len(translated):,} rows")
print(f"review    : {len(review):,} rows")
if review:
    reasons: dict[str, int] = {}
    for row in review:
        reasons[row["reason"]] = reasons.get(row["reason"], 0) + 1
    print("review reasons:", json.dumps(reasons, ensure_ascii=False, indent=2))
if translated:
    print("first row:", json.dumps(translated[0], ensure_ascii=False)[:600])

In [ ]:
# Stage 5 — Merge gold gốc + bản dịch
#
# Nguyên tắc: mọi field của record gốc giữ nguyên (cui, icd10, label, vi_term,
# vi_start_token_idx, vi_end_token_idx, vi_sentence_str, ...); phần dịch được
# thêm vào dưới tên en_*:
#   entities[].vi_term            (bản dịch) -> entities[].en_term
#   entities[].vi_start_token_idx (bản dịch) -> entities[].en_start_token_idx
#   entities[].vi_end_token_idx   (bản dịch) -> entities[].en_end_token_idx
#   vi_sentence_str               (bản dịch) -> en_sentence_str
#
# Ghép theo manifest (.done.jsonl), không theo thứ tự dòng: translated.jsonl
# bỏ qua record fail nên dòng thứ i không tương ứng record thứ i của input.

import hashlib

MERGED_PATH = OUTPUT_DIR / "merged.jsonl"


def _read_raw_lines(path) -> list[str]:
    path = Path(path)
    if not path.is_file():
        return []
    with path.open(encoding="utf-8-sig") as handle:
        return [line.rstrip("\r\n") for line in handle if line.strip()]


def read_jsonl(path) -> list[dict]:
    return [json.loads(line) for line in _read_raw_lines(path)]


raw_lines = _read_raw_lines(INPUT_PATH)
records = [json.loads(line) for line in raw_lines]
translated = read_jsonl(OUTPUT_PATH)
manifest = read_jsonl(str(OUTPUT_PATH) + ".done.jsonl")

assert len(translated) == len(manifest), (
    f"{len(translated)} dòng dịch nhưng {len(manifest)} dòng manifest"
)

rows: list[tuple[int, dict]] = []
for out_row, entry in zip(translated, manifest):
    index = entry["index"]
    assert hashlib.sha1(raw_lines[index].encode("utf-8")).hexdigest() == entry["sha1"], (
        f"record {index}: manifest sha1 không khớp input"
    )
    source = records[index]
    assert len(source["entities"]) == len(out_row["entities"]), (
        f"record {index}: lệch số entity"
    )
    entities = []
    for src_e, tr_e in zip(source["entities"], out_row["entities"]):
        assert src_e["cui"] == tr_e["cui"], f"record {index}: lệch thứ tự cui"
        entities.append(
            {
                **src_e,
                "en_term": tr_e["vi_term"],
                "en_start_token_idx": tr_e["vi_start_token_idx"],
                "en_end_token_idx": tr_e["vi_end_token_idx"],
            }
        )
    rows.append(
        (
            index,
            {
                **source,
                "entities": entities,
                "en_sentence_str": out_row["vi_sentence_str"],
            },
        )
    )

rows.sort(key=lambda item: item[0])
with MERGED_PATH.open("w", encoding="utf-8") as handle:
    for _index, row in rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"gốc       : {INPUT_PATH} ({len(records):,} records)")
print(f"dịch      : {OUTPUT_PATH} ({len(translated):,} records)")
print(f"merge     : {MERGED_PATH} ({len(rows):,} records)")
print(f"chưa dịch : {len(records) - len(rows):,} records (xem review.jsonl)")
if rows:
    print("first merged row:", json.dumps(rows[0][1], ensure_ascii=False)[:600])